# Week 9 - Exploratory Data Analysis

**Paññāsāstra University of Cambodia**
**Course:** Data Analysis and Visualization (3 credits)
**Chapter 9 - Exploratory Data Analysis**

---

## Session topics (from the course outline)

- Understanding datasets through visualization
- Correlation and covariance
- Detecting outliers

## Course learning outcomes addressed

| CLO | Statement |
|---|---|
| **CLO 3** | Apply Python libraries (NumPy, Pandas) to clean, manipulate and process datasets |
| **CLO 4** | Use visualization tools to create charts, graphs and plots for effective data presentation |
| **CLO 5** | Perform exploratory data analysis (EDA) to summarize and interpret datasets |
| **CLO 8** | Evaluate the effectiveness of different visualization methods for communicating insights |
| **CLO 9** | Design interactive dashboards or visual reports to support data-driven decisions |

## Reading

| Topic | Source |
|---|---|
| Summarizing and computing descriptive statistics | McKinney §5.3 |
| Correlation and covariance | McKinney §5.3 |
| Detecting and filtering outliers | McKinney §7.2 |
| Descriptive statistics | Pfaffelhuber §9.1 |

## Putting the last four weeks together

This session has little new syntax. Everything in it uses tools from Weeks 5 to 8: cleaning,
grouping, matplotlib and seaborn. What it adds is a **procedure** - a systematic first pass
over an unfamiliar dataset and a set of specific traps that catch people during it.

The central one is **Simpson's paradox**, in §2.5: a relationship that is positive in every
subgroup and negative overall. It is not a curiosity. It has decided court cases and medical
guidelines, and it is the strongest argument in this course for why you group before you
conclude.

---

## Setup

All imports for this notebook are defined in the single cell below. Run it first. If any
import fails, return to Notebook 00 and check that the kernel in the top-right corner points
inside your `venv`.

In [ ]:
# ---------------------------------------------------------------------------
# All imports for this notebook. Run this cell first.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Reproducibility: every dataset in this notebook is generated from this seed,
# so your numbers will match the ones shown in the lecture exactly.
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Consistent, colour-blind-safe plotting defaults for the whole course.
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860"]
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 100,
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.alpha": 0.3,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "font.size": 10,
})
sns.set_palette(PALETTE)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("Environment ready.")
print(f"  numpy   {np.__version__}")
print(f"  pandas  {pd.__version__}")
print(f"  seaborn {sns.__version__}")
print(f"  seed    {RANDOM_SEED}")

---

# Part 1 — Theory and concepts

## 1.1 What EDA is

Exploratory data analysis is the stage between having clean data and having a model. Its
purpose is to find out what you have before you commit to a claim.

A workable order, and the one this session follows:

| Step | Question | Tool |
|---|---|---|
| 1 | How much data, and of what types? | `.shape`, `.info()` |
| 2 | What is missing? | `.isna().sum()` |
| 3 | What does each variable look like alone? | histogram, `.describe()` |
| 4 | Are there impossible or extreme values? | five-number summary, Tukey fences |
| 5 | How do variables relate? | correlation matrix, scatter plots |
| 6 | **Does the relationship hold within subgroups?** | `groupby`, coloured scatter |

Step 6 is the one people skip, and it is the one that matters most.

## 1.2 Covariance and correlation

Covariance measures how two variables move together:

$$\mathrm{cov}(x, y) = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})$$

It is positive when they tend to rise together and negative when one rises as the other falls.
Its problem is units: the covariance of height and weight is in centimetre-kilograms, so its
size tells you nothing on its own.

Correlation divides that by the two standard deviations, giving a unitless number in
$[-1, 1]$:

$$r_{xy} = \frac{\mathrm{cov}(x, y)}{s_x s_y}$$

This is Pearson's $r$, which you met in Week 1. **It measures linear association only.**

## 1.3 Pearson and Spearman

| Coefficient | Measures | Robust to outliers | Detects |
|---|---|:---:|---|
| **Pearson** | Linear association | no | straight-line relationships |
| **Spearman** | Monotone association | yes | any consistently increasing or decreasing relationship |

Spearman's $\rho$ is Pearson's $r$ computed on the **ranks** rather than the values. Because
ranks are unaffected by how extreme a value is, one wild observation moves Spearman very
little and Pearson a great deal.

**Comparing the two is a cheap diagnostic.** If Spearman is much larger than Pearson, the
relationship is probably monotone but curved, or is being pulled about by outliers. If they
agree, the relationship is roughly linear. `df.corr(method="spearman")` is one argument away.

## 1.4 The four traps

A correlation coefficient is one number describing a two-dimensional relationship. Four things
it can hide:

1. **Non-linearity.** Anscombe's dataset II: a clean curve with $r = 0.816$. A near-zero
   correlation does not mean "no relationship"; it means "no *linear* relationship". A
   U-shaped relationship has $r \approx 0$.
2. **Outliers.** Anscombe's III and IV: a single point creating or destroying the
   correlation. Compare Pearson with Spearman; recompute without the point and see what moves.
3. **Confounding.** Both variables driven by a third. Ice cream sales and drowning deaths
   correlate because both rise in summer.
4. **Simpson's paradox.** The relationship reverses when you condition on a group. This is
   §2.5, and it is the most alarming of the four because both the aggregate and the subgroup
   results are arithmetically correct.

## 1.5 Simpson's paradox

Simpson's paradox occurs when a trend present in every subgroup disappears or reverses when
the groups are combined.

The classic real example is the 1973 Berkeley graduate admissions data. Men were admitted at a
higher rate than women overall. Examined department by department, most departments admitted
women at a *higher* rate than men. Both statements are true of the same data.

The explanation is that departments differed in how selective they were, and women applied in
greater numbers to the more competitive departments. Department is a **confounder**: it
affects both the application pattern and the admission rate.

Which number is the right one depends on the question. "What proportion of women who applied
were admitted?" is answered by the aggregate. "Was the admissions process biased?" is answered
by the within-department figures. The aggregate is not wrong; it answers a different question
than the one people assume.

**The practical defence is step 6 of §1.1.** Whenever you report a relationship, check whether
it survives conditioning on the obvious groups. If it does not, the grouping variable is part
of the story.

## 1.6 Detecting outliers

Two standard rules.

**Tukey's fences**, from Week 1's homework, based on quartiles:

$$x < Q_1 - 1.5\,\mathrm{IQR} \qquad \text{or} \qquad x > Q_3 + 1.5\,\mathrm{IQR}$$

**The z-score rule**, based on the mean and standard deviation:

$$|z| = \left|\frac{x - \bar{x}}{s}\right| > 3$$

They disagree, and the reason matters. The z-score rule uses the mean and standard deviation,
**both of which the outlier itself inflates**. One extreme value drags the mean towards itself
and enlarges $s$, which widens the threshold, which can hide the very point you are looking
for. This is called masking. Tukey's rule uses quartiles, which a single extreme value barely
moves, so it does not suffer from it.

**Prefer Tukey's rule** unless you have a specific reason for the z-score, and remember that
both are conventions calibrated to roughly normal data. On a skewed distribution Tukey's rule
flags a long tail that is entirely genuine.

### What to do with one

Detecting an outlier is not a decision. The decision depends on what it is:

| Kind | Example | Action |
|---|---|---|
| **Data error** | An age of 999, a negative price | Correct it, or set it to missing |
| **Different population** | A wholesale order in a retail dataset | Exclude it, and say so |
| **Genuine extreme** | A genuinely exceptional student | **Keep it.** It is the data |

Deleting a point because it is inconvenient is not cleaning. Every exclusion needs a stated
reason, and the count of excluded points belongs in your report.

---

---

# Part 2 — Code example

§2.5 is the centre of the session.

## 2.1 The dataset

In [ ]:
def generate_admissions(seed: int = RANDOM_SEED) -> pd.DataFrame:
    """Build a dataset that contains Simpson's paradox by construction.

    Four departments differ in selectivity. Applicants of type X apply mostly
    to the easy departments, type Y mostly to the hard ones. Within EVERY
    department, type Y is admitted at a higher rate. In aggregate, type X is.

    Returns
    -------
    pandas.DataFrame
        One row per applicant.
    """
    generator = np.random.default_rng(seed)

    # (department, admission rate for X, admission rate for Y, n of X, n of Y)
    departments = [
        ("Literature", 0.62, 0.72, 320, 40),
        ("Education",  0.55, 0.65, 260, 60),
        ("Physics",    0.22, 0.32,  50, 250),
        ("Computing",  0.16, 0.26,  30, 300),
    ]

    rows = []
    for name, rate_x, rate_y, n_x, n_y in departments:
        for applicant_type, rate, count in [("X", rate_x, n_x), ("Y", rate_y, n_y)]:
            # Admit an EXACT number rather than drawing at random, so the rates
            # are precisely the ones in the table above. Sampling noise on a
            # group of 30 would otherwise shift a rate by several points and
            # blur the very comparison this dataset exists to show.
            n_admitted = int(round(rate * count))
            rows.extend({
                "department": name,
                "applicant_type": applicant_type,
                "admitted": index < n_admitted,
            } for index in range(count))

    frame = pd.DataFrame(rows)
    return frame.sample(frac=1, random_state=seed).reset_index(drop=True)


def generate_cohort(n: int = 400, seed: int = RANDOM_SEED) -> pd.DataFrame:
    """Build a student cohort with a deliberate outlier and a curved relationship."""
    generator = np.random.default_rng(seed)

    faculties = ["Engineering", "Business", "Science", "Arts"]
    faculty = generator.choice(faculties, size=n, p=[0.35, 0.30, 0.20, 0.15])
    ability = generator.normal(0, 1, size=n)

    study = np.clip(generator.normal(9, 3.2, n) + ability * 1.1, 0.5, None).round(1)

    # An inverted-U: performance rises with study hours and then falls, because
    # very long hours come with exhaustion. Pearson will barely see this.
    curved = 55 + 6.5 * study - 0.32 * study ** 2 + generator.normal(0, 4, n)

    frame = pd.DataFrame({
        "student_id": [f"S{2026000 + i}" for i in range(n)],
        "faculty": faculty,
        "study_hours": study,
        "exam_score": np.clip(curved, 0, 100).round(1),
        "gpa": np.clip(2.6 + ability * 0.5, 0, 4).round(2),
        "attendance_pct": np.clip(78 + ability * 9 + generator.normal(0, 5, n),
                                  0, 100).round(1),
        "library_visits": generator.integers(0, 60, size=n),
    })

    # One data-entry error: a decimal point in the wrong place.
    frame.loc[7, "library_visits"] = 580

    return frame


admissions = generate_admissions()
cohort = generate_cohort()

print(f"admissions: {admissions.shape[0]} applicants")
print(f"cohort    : {cohort.shape[0]} students")
print()
print(admissions.head())

## 2.2 Steps 1 to 3: shape, missingness, one variable at a time

In [ ]:
print("STEP 1 — how much data, and of what types?")
print("=" * 62)
cohort.info()

print("\nSTEP 2 — what is missing?")
print("=" * 62)
missing = cohort.isna().sum()
print("no missing values" if missing.sum() == 0 else missing[missing > 0])

print("\nSTEP 3 — what does each variable look like alone?")
print("=" * 62)
print(cohort.describe().round(2).to_string())

In [ ]:
numeric_columns = ["study_hours", "exam_score", "gpa", "attendance_pct", "library_visits"]

fig, axes = plt.subplots(1, 5, figsize=(17, 3.2))
for ax, column in zip(axes, numeric_columns):
    ax.hist(cohort[column], bins=22, color=PALETTE[0], edgecolor="white")
    ax.set_title(column, loc="left", fontsize=10)
    ax.tick_params(labelsize=8)
axes[0].set_ylabel("students")

fig.suptitle("Step 3: every numeric variable, alone", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("Four of the five look unremarkable. library_visits does not: almost")
print("everything is squeezed against the left edge because a single value is")
print("far out to the right, stretching the axis.")
print("\nThat is what an outlier looks like in a histogram — not a visible")
print("point, but a distribution that has been squashed.")

## 2.3 Step 4: outliers

Two rules, and the reason they disagree.

In [ ]:
def tukey_outliers(values) -> dict:
    """Flag outliers with Tukey's 1.5 x IQR rule."""
    x = np.asarray(values, dtype=float)
    x = x[~np.isnan(x)]
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (x < lower) | (x > upper)
    return {"lower": lower, "upper": upper, "n": int(mask.sum()),
            "values": np.sort(x[mask]).tolist()}


def zscore_outliers(values, threshold: float = 3.0) -> dict:
    """Flag outliers with the |z| > threshold rule."""
    x = np.asarray(values, dtype=float)
    x = x[~np.isnan(x)]
    mean, std = x.mean(), x.std(ddof=1)
    z = (x - mean) / std
    mask = np.abs(z) > threshold
    return {"lower": mean - threshold * std, "upper": mean + threshold * std,
            "n": int(mask.sum()), "values": np.sort(x[mask]).tolist()}


visits = cohort["library_visits"]
tukey = tukey_outliers(visits)
zscore = zscore_outliers(visits)

print(f"library_visits: n = {len(visits)}, "
      f"median = {visits.median():.0f}, mean = {visits.mean():.1f}")
print()
print(f"{'rule':<10}{'lower':>10}{'upper':>10}{'n flagged':>12}   values")
print("-" * 64)
print(f"{'Tukey':<10}{tukey['lower']:>10.1f}{tukey['upper']:>10.1f}"
      f"{tukey['n']:>12}   {tukey['values']}")
print(f"{'z > 3':<10}{zscore['lower']:>10.1f}{zscore['upper']:>10.1f}"
      f"{zscore['n']:>12}   {zscore['values']}")

In [ ]:
# Both rules caught this one. But look at how far apart the fences are,
# and at what the single outlier did to the statistics the z-rule depends on.
without = visits[visits < 500]

print("Effect of that ONE value on each rule's ingredients:")
print(f"{'':<12}{'with it':>12}{'without it':>13}{'change':>10}")
print("-" * 47)
print(f"{'mean':<12}{visits.mean():>12.2f}{without.mean():>13.2f}"
      f"{(visits.mean() / without.mean() - 1) * 100:>9.0f}%")
print(f"{'std':<12}{visits.std(ddof=1):>12.2f}{without.std(ddof=1):>13.2f}"
      f"{(visits.std(ddof=1) / without.std(ddof=1) - 1) * 100:>9.0f}%")
print(f"{'Q3':<12}{np.percentile(visits, 75):>12.2f}"
      f"{np.percentile(without, 75):>13.2f}"
      f"{(np.percentile(visits, 75) / np.percentile(without, 75) - 1) * 100:>9.0f}%")

print("\nOne value in 400 barely moved the quartiles and enlarged the standard")
print("deviation by most of its own size again.")
print(f"\nThat is why the fences differ so much: Tukey's upper fence is "
      f"{tukey['upper']:.0f}, the z-score's is {zscore['upper']:.0f}.")
print("The z-score rule has already been pushed outwards by the very value")
print("it is trying to find. Here it still catches it — 580 is far enough out")
print("that even the inflated fence falls short of it.")
print("\nThe rule fails when the outliers are closer in, or when the sample is")
print("small. The next cell shows that happening.")

In [ ]:
# MASKING: where the z-score rule fails outright.
# A single tutorial group of 12 students, two of whom are heavy library users.
tutorial = pd.Series([4, 6, 7, 8, 9, 11, 12, 13, 15, 18, 95, 110])

t = tukey_outliers(tutorial)
z = zscore_outliers(tutorial)

print("Library visits for one tutorial group of 12:")
print(f"  {sorted(tutorial)}")
print()
print(f"{'rule':<10}{'upper fence':>14}{'n flagged':>12}")
print("-" * 36)
print(f"{'Tukey':<10}{t['upper']:>14.1f}{t['n']:>12}")
print(f"{'z > 3':<10}{z['upper']:>14.1f}{z['n']:>12}")

quiet = tutorial[tutorial < 50]
print(f"\nStandard deviation of the other ten students : {quiet.std(ddof=1):.1f}")
print(f"Standard deviation with the two included     : {tutorial.std(ddof=1):.1f}")

print("\nThe two heavy users inflated the standard deviation eightfold. Three")
print(f"of those is {3 * tutorial.std(ddof=1):.0f}, so the z-score fence lands at "
      f"{z['upper']:.0f} —")
print("above BOTH of them. The rule designed to find them has been pushed out")
print("of range by their own presence, and it reports that nothing is unusual.")
print("\nThis is MASKING, and it is why a quartile-based rule is the safer")
print("default. Tukey's fences moved hardly at all, and flagged both.")

In [ ]:
# Detecting is not deciding.
print("What IS this value?")
print("=" * 62)
print(cohort.loc[[7], ["student_id", "faculty", "library_visits", "exam_score"]]
      .to_string(index=False))
print(f"\nSecond highest value in the column: {visits.nlargest(2).iloc[1]}")
print(f"The flagged value is {visits.max() / visits.nlargest(2).iloc[1]:.0f}x larger.")

print("\n580 library visits in one semester is roughly four per day, every day,")
print("including weekends. The next highest student managed 59. This is a")
print("decimal point in the wrong place — a DATA ERROR, and the right action")
print("is to correct it or mark it missing.")

print("\nHad it been 59 rather than 580, Tukey's rule would still have flagged")
print("it, and the right action would have been to KEEP it: a genuinely")
print("diligent student is data, not noise.")
print("\nThe rule finds candidates. You decide what they are.")

## 2.4 Step 5: how variables relate

In [ ]:
clean = cohort.copy()
clean.loc[7, "library_visits"] = np.nan          # the data error, removed

pearson = clean[numeric_columns].corr(method="pearson")
spearman = clean[numeric_columns].corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
mask = np.triu(np.ones_like(pearson, dtype=bool))

for ax, matrix, name in zip(axes, [pearson, spearman], ["Pearson", "Spearman"]):
    sns.heatmap(matrix, annot=True, fmt=".2f", ax=ax, mask=mask,
                cmap="RdBu_r", vmin=-1, vmax=1, center=0, linewidths=0.5,
                annot_kws={"size": 8}, cbar=False)
    ax.set_title(f"{name}", loc="left")
    ax.tick_params(labelsize=7)

fig.suptitle("The same data, two correlation measures",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# The diagonal is |1 - 1| = 0 in both matrices, so it can never be the
# largest disagreement and needs no special handling.
difference = (spearman - pearson).abs()
worst = difference.stack().idxmax()
print(f"Largest disagreement: {worst[0]} and {worst[1]}")
print(f"  Pearson  {pearson.loc[worst]:+.3f}")
print(f"  Spearman {spearman.loc[worst]:+.3f}")
print("\nA large gap between the two is a signal: the relationship is monotone")
print("but not linear, or it is being pulled by extreme values. Either way,")
print("go and look at the scatter plot.")

In [ ]:
# The curved relationship the correlation coefficient cannot see.
r_pearson = clean["study_hours"].corr(clean["exam_score"])
r_spearman = clean["study_hours"].corr(clean["exam_score"], method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

axes[0].scatter(clean["study_hours"], clean["exam_score"], s=18, alpha=0.5,
                color=PALETTE[0], edgecolor="none")
slope, intercept = np.polyfit(clean["study_hours"], clean["exam_score"], 1)
line_x = np.linspace(clean["study_hours"].min(), clean["study_hours"].max(), 50)
axes[0].plot(line_x, intercept + slope * line_x, color=PALETTE[3],
             linewidth=2, label="straight line")
axes[0].set_title(f"(a) A straight line: r = {r_pearson:.3f}", loc="left")
axes[0].set_xlabel("study hours per week")
axes[0].set_ylabel("exam score")
axes[0].legend(fontsize=8, frameon=False)

axes[1].scatter(clean["study_hours"], clean["exam_score"], s=18, alpha=0.5,
                color=PALETTE[0], edgecolor="none")
quad = np.polyfit(clean["study_hours"], clean["exam_score"], 2)
axes[1].plot(line_x, np.polyval(quad, line_x), color=PALETTE[2],
             linewidth=2, label="quadratic")
axes[1].set_title("(b) A curve", loc="left")
axes[1].set_xlabel("study hours per week")
axes[1].set_ylabel("exam score")
axes[1].legend(fontsize=8, frameon=False)

plt.tight_layout()
plt.show()

peak = -quad[1] / (2 * quad[0])
print(f"Pearson r  = {r_pearson:+.3f}   Spearman rho = {r_spearman:+.3f}")
print(f"\nThe fitted curve peaks at {peak:.1f} hours per week and falls after that.")
print("\nA correlation near zero would have read as 'study hours do not matter'.")
print("What the data says is 'they matter, and there is an optimum'. The")
print("coefficient could not express that, because it only measures how well")
print("a STRAIGHT LINE fits.")

## 2.5 Step 6: does it hold within subgroups?

Simpson's paradox. The step people skip.

In [ ]:
overall = admissions.groupby("applicant_type")["admitted"].agg(
    applicants="count", admitted="sum"
)
overall["rate_pct"] = (overall["admitted"] / overall["applicants"] * 100).round(1)

print("ADMISSION RATES, OVERALL")
print("=" * 44)
print(overall.to_string())

gap = overall.loc["X", "rate_pct"] - overall.loc["Y", "rate_pct"]
print(f"\nType X is admitted at a rate {gap:.1f} percentage points higher than type Y.")
print("\nOn this table, the process favours type X.")

In [ ]:
by_department = admissions.groupby(["department", "applicant_type"])["admitted"].agg(
    applicants="count", admitted="sum"
)
by_department["rate_pct"] = (
    by_department["admitted"] / by_department["applicants"] * 100
).round(1)

wide = by_department["rate_pct"].unstack()
wide["Y minus X"] = (wide["Y"] - wide["X"]).round(1)

print("ADMISSION RATES, BY DEPARTMENT")
print("=" * 52)
print(wide.to_string())

print("\nIn EVERY department, type Y is admitted at a higher rate than type X.")
print("\nBoth tables are arithmetically correct, computed from the same rows.")

In [ ]:
# Why: the two groups applied to different departments.
applications = admissions.groupby(["department", "applicant_type"]).size().unstack()
share = (applications.div(applications.sum(axis=1), axis=0) * 100).round(1)

difficulty = (admissions.groupby("department")["admitted"].mean() * 100).round(1)

explain = pd.DataFrame({
    "overall_rate_pct": difficulty,
    "n_X": applications["X"],
    "n_Y": applications["Y"],
    "pct_of_applicants_Y": share["Y"],
}).sort_values("overall_rate_pct", ascending=False)

print("THE EXPLANATION")
print("=" * 66)
print(explain.to_string())

print("\nThe departments differ enormously in selectivity: from about "
      f"{difficulty.max():.0f}% down to {difficulty.min():.0f}%.")
print("\nType X applied overwhelmingly to the easy departments. Type Y applied")
print("overwhelmingly to the hard ones. So type Y's overall rate is dragged")
print("down by WHERE they applied, not by how they were treated once there.")
print("\nDepartment is a CONFOUNDER: it affects both who applies and who is")
print("admitted.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# (a) the aggregate
axes[0].bar(overall.index, overall["rate_pct"],
            color=[PALETTE[0], PALETTE[1]], edgecolor="white", width=0.5)
axes[0].set_ylim(0, 70)
axes[0].set_title("(a) Overall: X does better", loc="left")
axes[0].set_ylabel("admission rate (%)")
axes[0].set_xlabel("applicant type")
for i, value in enumerate(overall["rate_pct"]):
    axes[0].text(i, value + 1.5, f"{value:.1f}%", ha="center", fontsize=9)

# (b) by department
order = difficulty.sort_values(ascending=False).index
positions = np.arange(len(order))
axes[1].bar(positions - 0.19, wide.loc[order, "X"], width=0.38,
            color=PALETTE[0], edgecolor="white", label="X")
axes[1].bar(positions + 0.19, wide.loc[order, "Y"], width=0.38,
            color=PALETTE[1], edgecolor="white", label="Y")
axes[1].set_xticks(positions, order, rotation=20, fontsize=8)
axes[1].set_ylim(0, 70)
axes[1].set_title("(b) By department: Y does better, every time", loc="left")
axes[1].set_ylabel("admission rate (%)")
axes[1].legend(fontsize=8, frameon=False)

# (c) where each group applied
axes[2].bar(positions - 0.19, applications.loc[order, "X"], width=0.38,
            color=PALETTE[0], edgecolor="white", label="X")
axes[2].bar(positions + 0.19, applications.loc[order, "Y"], width=0.38,
            color=PALETTE[1], edgecolor="white", label="Y")
axes[2].set_xticks(positions, order, rotation=20, fontsize=8)
axes[2].set_title("(c) Why: they applied to different departments", loc="left")
axes[2].set_ylabel("applicants")
axes[2].legend(fontsize=8, frameon=False)

fig.suptitle("Simpson's paradox: both panels are correct",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("Panel (b) is sorted by how easy the department is, left to right.")
print("Panel (c) shows type X clustered in the easy departments on the left")
print("and type Y in the hard ones on the right.")

In [ ]:
print("WHICH NUMBER IS RIGHT?")
print("=" * 66)
print("Both. They answer different questions.")
print()
print('  "What proportion of type Y applicants were admitted?"')
print(f'      -> the aggregate: {overall.loc["Y", "rate_pct"]:.1f}%')
print()
print('  "Were type Y applicants treated worse by the admissions process?"')
print("      -> the by-department figures: no, they were treated better")
print("         in every department")
print()
print("The aggregate is not a mistake. It is the answer to a question people")
print("then misread as being about fairness.")
print()
print("This is the 1973 Berkeley graduate admissions case, which is the")
print("standard real example: men were admitted at a higher rate overall,")
print("women at a higher rate in most individual departments.")
print()
print("=" * 66)
print("THE PRACTICAL RULE")
print("=" * 66)
print("  Before you report a relationship, check whether it survives")
print("  conditioning on the obvious groups. If it does not, the grouping")
print("  variable is part of the story and belongs in the report.")

## 2.6 The whole pass, as one function

In [ ]:
def explore(df: pd.DataFrame, group_col: str = None) -> dict:
    """Run the six-step first pass over a DataFrame and return the findings.

    Parameters
    ----------
    df : pandas.DataFrame
        The data. Not modified.
    group_col : str or None
        If given, correlations are also computed within each group, so that
        a Simpson's-paradox reversal becomes visible.

    Returns
    -------
    dict
        Findings from each step.
    """
    numeric = df.select_dtypes(include="number")

    outliers = {}
    for column in numeric.columns:
        found = tukey_outliers(numeric[column])
        if found["n"]:
            outliers[column] = found["n"]

    findings = {
        "shape": df.shape,
        "missing_total": int(df.isna().sum().sum()),
        "numeric_columns": list(numeric.columns),
        "outlier_counts": outliers,
        "pearson": numeric.corr(),
        "spearman": numeric.corr(method="spearman"),
    }

    # Step 6: does each correlation survive conditioning on the group?
    if group_col is not None and group_col in df.columns:
        within = {}
        for name, group in df.groupby(group_col, observed=True):
            if len(group) > 2:
                within[name] = group[numeric.columns].corr()
        findings["within_group_pearson"] = within

    return findings


results = explore(clean, group_col="faculty")

print(f"shape            : {results['shape']}")
print(f"missing          : {results['missing_total']}")
print(f"outliers (Tukey) : {results['outlier_counts']}")
print(f"groups examined  : {sorted(results['within_group_pearson'])}")

print("\nattendance/gpa correlation, overall and within each faculty:")
print(f"  {'overall':<14}{results['pearson'].loc['attendance_pct', 'gpa']:+.3f}")
for name, matrix in sorted(results["within_group_pearson"].items()):
    print(f"  {name:<14}{matrix.loc['attendance_pct', 'gpa']:+.3f}")

print("\nHere the relationship holds within every faculty, at about the same")
print("strength. That is the reassuring outcome, and it is worth reporting")
print("explicitly — 'we checked, and it holds' is a finding.")

---

# Part 3 — Guided workshop (in class)

Four tasks. Each has a verification cell.

### Task 1 — Compare the two correlation measures

**Problem.** Compute `pearson_r` and `spearman_r` for `study_hours` against `exam_score` in
`clean`, then set `bigger` to `"spearman"` or `"pearson"` — whichever is larger in absolute
value.

**Hints**

- `series.corr(other)` gives Pearson; `series.corr(other, method="spearman")` gives Spearman.
- `abs()` for the comparison.

**Expected output.** Two coefficients and the answer, followed by `Task 1 complete`.

In [ ]:
# TODO 1a: Pearson correlation of study_hours and exam_score.
pearson_r = None

# TODO 1b: Spearman correlation of the same pair.
spearman_r = None

# TODO 1c: which is larger in absolute value? "pearson" or "spearman"
bigger = None

print(f"Pearson  : {pearson_r}")
print(f"Spearman : {spearman_r}")
print(f"Larger   : {bigger}")

In [ ]:
# Verification for Task 1.
want_p = clean["study_hours"].corr(clean["exam_score"])
want_s = clean["study_hours"].corr(clean["exam_score"], method="spearman")
want_bigger = "spearman" if abs(want_s) > abs(want_p) else "pearson"

problems = []
if pearson_r is None or not np.isclose(float(pearson_r), want_p):
    problems.append(f"pearson_r should be about {want_p:.4f}")
if spearman_r is None or not np.isclose(float(spearman_r), want_s):
    problems.append(f"spearman_r should be about {want_s:.4f}")
if bigger is None or str(bigger).lower() != want_bigger:
    problems.append(f"bigger should be {want_bigger!r}")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print("Task 1 complete.")
    print(f"\nBoth are small, because the relationship is a CURVE. Neither")
    print(f"coefficient can express 'rises then falls' — which is why step 5")
    print(f"is followed by looking at the scatter plot, not replaced by it.")

### Task 2 — Find the outliers

**Problem.** Using `tukey_outliers` from §2.3, find the outliers in `attendance_pct`.

Store the returned dictionary in `attendance_outliers`, and set `n_attendance_outliers` to
the count.

Then set `visits_differ` to `True` or `False`: do Tukey's rule and the z-score rule flag
*different numbers* of outliers in the original `cohort["library_visits"]`?

**Hints**

- Both functions are already defined in §2.3.
- Compare the `"n"` entry of each result.

**Expected output.** A count and a boolean, followed by `Task 2 complete`.

In [ ]:
# TODO 2a: Tukey outliers in attendance_pct.
attendance_outliers = None
n_attendance_outliers = None

# TODO 2b: do the two rules disagree on library_visits? True or False.
visits_differ = None

print(f"attendance_pct outliers : {n_attendance_outliers}")
print(f"the two rules disagree  : {visits_differ}")

In [ ]:
# Verification for Task 2.
want = tukey_outliers(cohort["attendance_pct"])
want_differ = (tukey_outliers(cohort["library_visits"])["n"]
               != zscore_outliers(cohort["library_visits"])["n"])

problems = []
if attendance_outliers is None:
    problems.append("attendance_outliers is still None")
elif not isinstance(attendance_outliers, dict):
    problems.append("attendance_outliers should be the dict the function returns")
elif attendance_outliers["n"] != want["n"]:
    problems.append(f"expected {want['n']} outliers, got {attendance_outliers['n']}")

if n_attendance_outliers is None or int(n_attendance_outliers) != want["n"]:
    problems.append(f"n_attendance_outliers should be {want['n']}")

if visits_differ is None:
    problems.append("visits_differ is still None")
elif bool(visits_differ) != want_differ:
    problems.append(f"visits_differ should be {want_differ}")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print(f"Task 2 complete. {want['n']} attendance outliers.")
    print(f"\nOn library_visits, Tukey flags "
          f"{tukey_outliers(cohort['library_visits'])['n']} and the z-score rule "
          f"flags {zscore_outliers(cohort['library_visits'])['n']}.")

### Task 3 — Check whether a relationship survives grouping

**Problem.** The correlation between `attendance_pct` and `gpa` in `clean` is strong. Check
whether it holds within each faculty.

Build `within_faculty`: a `Series` indexed by faculty, holding the correlation between
`attendance_pct` and `gpa` **within** that faculty.

Then set `reverses` to `True` if any faculty's correlation has the opposite sign to the
overall one, and `False` otherwise.

**Hints**

- `df.groupby(col).apply(lambda g: g["a"].corr(g["b"]))` gives one value per group. Pass
  `include_groups=False` if pandas warns.
- Compare `np.sign()` of each group value against the overall.

**Expected output.** Four correlations and a boolean, followed by `Task 3 complete`.

In [ ]:
overall_r = clean["attendance_pct"].corr(clean["gpa"])

# TODO 3a: the same correlation, computed within each faculty.
within_faculty = None

# TODO 3b: does any faculty reverse the sign? True or False.
reverses = None

print(f"overall: {overall_r:+.3f}\n")
print(within_faculty)
print(f"\nany sign reversal: {reverses}")

In [ ]:
# Verification for Task 3.
want = clean.groupby("faculty")[["attendance_pct", "gpa"]].apply(
    lambda g: g["attendance_pct"].corr(g["gpa"])
)
want_reverses = bool((np.sign(want) != np.sign(overall_r)).any())

problems = []
if within_faculty is None:
    problems.append("within_faculty is still None")
elif not isinstance(within_faculty, pd.Series):
    problems.append(f"within_faculty should be a Series, got "
                    f"{type(within_faculty).__name__}")
elif len(within_faculty) != 4:
    problems.append(f"expected 4 faculties, got {len(within_faculty)}")
elif not np.allclose(within_faculty.sort_index().to_numpy(),
                     want.sort_index().to_numpy()):
    problems.append("the values do not match the per-faculty correlations")

if reverses is None:
    problems.append("reverses is still None")
elif bool(reverses) != want_reverses:
    problems.append(f"reverses should be {want_reverses}")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print("Task 3 complete.")
    print("\nNo reversal here: the relationship holds in every faculty at")
    print("roughly the same strength. That is the reassuring answer, and")
    print("reporting that you checked is part of the analysis.")

### Task 4 — Reproduce the paradox

**Problem.** Using the `admissions` frame, compute both views yourself.

1. `rate_overall` — a `Series` indexed by `applicant_type` with the overall admission rate as
   a **proportion** (0 to 1).
2. `rate_by_dept` — a `DataFrame` indexed by `department` with one column per
   `applicant_type`, holding the within-department rates.
3. `y_better_everywhere` — `True` if type Y's rate exceeds type X's in **every** department.

**Hints**

- `admitted` is boolean, so its **mean** is the admission rate.
- `groupby(...)["admitted"].mean()` for the first.
- `groupby([...])["admitted"].mean().unstack()` for the second.
- `(df["Y"] > df["X"]).all()` for the third.

**Expected output.** Both tables and a boolean, followed by `Task 4 complete`.

In [ ]:
# TODO 4a: overall admission rate per applicant type, as a proportion.
rate_overall = None

# TODO 4b: admission rate per department per applicant type.
rate_by_dept = None

# TODO 4c: is Y's rate higher than X's in every department?
y_better_everywhere = None

print("Overall:")
print(rate_overall)
print("\nBy department:")
print(rate_by_dept)
print(f"\nY better in every department: {y_better_everywhere}")

In [ ]:
# Verification for Task 4.
want_overall = admissions.groupby("applicant_type")["admitted"].mean()
want_dept = admissions.groupby(["department", "applicant_type"])["admitted"].mean().unstack()
want_flag = bool((want_dept["Y"] > want_dept["X"]).all())

problems = []
if rate_overall is None:
    problems.append("rate_overall is still None")
elif not np.allclose(np.sort(np.asarray(rate_overall, dtype=float)),
                     np.sort(want_overall.to_numpy())):
    problems.append("rate_overall does not match — remember `admitted` is boolean, "
                    "so its MEAN is the rate")

if rate_by_dept is None:
    problems.append("rate_by_dept is still None")
elif not isinstance(rate_by_dept, pd.DataFrame):
    problems.append("rate_by_dept should be a DataFrame — use .unstack()")
elif rate_by_dept.shape != (4, 2):
    problems.append(f"expected 4 departments x 2 types, got {rate_by_dept.shape}")
elif not np.allclose(rate_by_dept.sort_index().to_numpy(),
                     want_dept.sort_index().to_numpy()):
    problems.append("the per-department rates do not match")

if y_better_everywhere is None:
    problems.append("y_better_everywhere is still None")
elif bool(y_better_everywhere) != want_flag:
    problems.append(f"y_better_everywhere should be {want_flag}")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    x_rate = want_overall["X"] * 100
    y_rate = want_overall["Y"] * 100
    print("Task 4 complete.")
    print(f"\n  Overall : X {x_rate:.1f}%  vs  Y {y_rate:.1f}%   -> X looks better")
    print(f"  Per dept: Y higher in all 4              -> Y is better")
    print("\n  Both computed from the same rows, by you, just now.")

---

# Part 4 — Homework

Run **Restart Kernel and Run All Cells** before submitting, then push to your
`puc-data-analysis` repository.

## Problem 1 — Flexible outlier detection

**Problem.** Write `find_outliers(values, method="tukey", threshold=None)`.

**Requirements.** Return a dictionary with exactly these keys:

| Key | Type | Value |
|---|---|---|
| `method` | `str` | The method used |
| `lower` | `float` | Lower cut-off |
| `upper` | `float` | Upper cut-off |
| `n_outliers` | `int` | How many values fall outside |
| `outliers` | `list` | Those values, **sorted ascending** |
| `pct` | `float` | Percentage of non-missing values flagged, rounded to 2 dp |

**The two methods**

| `method` | Cut-offs | Default `threshold` |
|---|---|---|
| `"tukey"` | $Q_1 - k\,\mathrm{IQR}$ and $Q_3 + k\,\mathrm{IQR}$ | `k = 1.5` |
| `"zscore"` | $\bar{x} \pm k s$, with `ddof=1` | `k = 3.0` |

**Requirements on behaviour**

- Discard missing values before computing anything.
- Raise `ValueError` for an unknown `method`.
- Raise `ValueError` if fewer than four non-missing values remain — quartiles are meaningless
  below that.
- Accept a list, a NumPy array or a pandas `Series`.
- Do not modify the input.

**Hints**

- `threshold=None` means "use the default for this method", so resolve it after checking which
  method was asked for.
- Both methods share the same final step: build a mask, sort, and count. Only the cut-offs
  differ — compute those in a branch and keep the rest common.
- `pct` is `n_outliers / n_valid * 100`, not a proportion.

In [ ]:
# Problem 1 — starter code.

def find_outliers(values, method: str = "tukey", threshold=None) -> dict:
    """Flag outliers by Tukey's fences or the z-score rule.

    Parameters
    ----------
    values : list, numpy.ndarray or pandas.Series
        Numeric observations. Missing values are discarded.
    method : str
        Either "tukey" or "zscore".
    threshold : float or None
        Multiplier. None uses 1.5 for tukey and 3.0 for zscore.

    Returns
    -------
    dict
        Keys: 'method', 'lower', 'upper', 'n_outliers', 'outliers', 'pct'.

    Raises
    ------
    ValueError
        For an unknown method, or fewer than four usable values.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete find_outliers()")


for method in ["tukey", "zscore"]:
    result = find_outliers(cohort["library_visits"], method=method)
    print(f"{result['method']:<8} fences [{result['lower']:.1f}, {result['upper']:.1f}]  "
          f"n={result['n_outliers']}  ({result['pct']}%)  {result['outliers']}")

In [ ]:
# Problem 1 — validation. All assertions must pass.

# Sorted: 1..9 then 100. Q1 = 3.25, Q3 = 7.75, IQR = 4.5
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 100]

tukey = find_outliers(data, method="tukey")

assert isinstance(tukey, dict), "find_outliers must return a dict"
expected_keys = {"method", "lower", "upper", "n_outliers", "outliers", "pct"}
assert set(tukey.keys()) == expected_keys, (
    f"Keys must be exactly {sorted(expected_keys)}; got {sorted(tukey.keys())}"
)

assert tukey["method"] == "tukey"
assert np.isclose(tukey["lower"], 3.25 - 1.5 * 4.5), "Q1 - 1.5*IQR"
assert np.isclose(tukey["upper"], 7.75 + 1.5 * 4.5), "Q3 + 1.5*IQR"
assert tukey["n_outliers"] == 1
assert tukey["outliers"] == [100.0]
assert np.isclose(tukey["pct"], 10.0), "1 of 10 is 10.0 percent, not 0.1"

# --- types are Python scalars ---
for key in ["lower", "upper", "pct"]:
    assert isinstance(tukey[key], float), f"{key} must be a Python float"
assert isinstance(tukey["n_outliers"], int), "n_outliers must be a Python int"
assert isinstance(tukey["outliers"], list), "outliers must be a list"

# --- the z-score rule gives different cut-offs on the same data ---
z = find_outliers(data, method="zscore")
assert z["method"] == "zscore"
reference = np.asarray(data, dtype=float)
assert np.isclose(z["upper"], reference.mean() + 3 * reference.std(ddof=1)), (
    "z-score cut-offs use the mean and the SAMPLE standard deviation (ddof=1)"
)
assert z["upper"] > tukey["upper"], (
    "On this data the z-score fence is wider, because the outlier inflated "
    "the standard deviation — the masking effect from section 2.3"
)
assert z["n_outliers"] == 0, (
    f"With one extreme value in ten, |z| never exceeds 3 — the outlier masked "
    f"itself. Got {z['n_outliers']}"
)

# --- an explicit threshold is honoured ---
strict = find_outliers(data, method="tukey", threshold=0.5)
assert strict["n_outliers"] >= tukey["n_outliers"], (
    "A smaller multiplier gives tighter fences and flags at least as many"
)
loose = find_outliers(data, method="zscore", threshold=1.0)
assert loose["n_outliers"] > z["n_outliers"], "A threshold of 1.0 flags more than 3.0"

# --- both tails are found, and the list is sorted ---
both = find_outliers([-60, 1, 2, 3, 4, 5, 6, 7, 8, 70], method="tukey")
assert -60.0 in both["outliers"] and 70.0 in both["outliers"]
assert both["outliers"] == sorted(both["outliers"]), "outliers must be sorted ascending"

# --- clean data flags nothing ---
none_found = find_outliers([10, 20, 30, 40, 50], method="tukey")
assert none_found["n_outliers"] == 0
assert none_found["outliers"] == []
assert np.isclose(none_found["pct"], 0.0)

# --- missing values are discarded, not propagated ---
with_nan = find_outliers([1, 2, np.nan, 3, 4, 5, 100], method="tukey")
assert not np.isnan(with_nan["lower"]), "NaN must be dropped before computing"
assert 100.0 in with_nan["outliers"]
assert np.isclose(with_nan["pct"], 100 / 6 * 100 / 100 * 100 / 100 * (1 / 6) * 100 / (1 / 6) / 100 * 100 / 6) or True
assert np.isclose(with_nan["pct"], round(1 / 6 * 100, 2)), (
    "pct must be computed over the NON-MISSING values only"
)

# --- all three input types agree ---
as_list = find_outliers([1, 2, 3, 4, 5, 100])
as_array = find_outliers(np.array([1, 2, 3, 4, 5, 100]))
as_series = find_outliers(pd.Series([1, 2, 3, 4, 5, 100]))
assert as_list == as_array == as_series

# --- errors ---
try:
    find_outliers(data, method="nonsense")
except ValueError:
    pass
else:
    raise AssertionError("An unknown method must raise ValueError")

for too_few in [[1.0, 2.0, 3.0], [], [np.nan, np.nan]]:
    try:
        find_outliers(too_few)
    except ValueError:
        pass
    else:
        raise AssertionError(f"find_outliers({too_few!r}) must raise ValueError")

# --- the input is untouched ---
source = pd.Series([1.0, 2.0, 3.0, 4.0, 100.0])
before = source.copy()
_ = find_outliers(source)
assert source.equals(before), "find_outliers must not modify its input"

print("Problem 1: all assertions passed.")

---

## Problem 2 — Test a relationship for reversal

Step 6 of the EDA procedure, written as a function.

**Problem.** Write `check_within_groups(df, x_col, y_col, group_col)`.

It computes the correlation between `x_col` and `y_col` overall and within each group, and
reports whether the relationship changes sign.

**Requirements.** Return a dictionary with exactly these keys:

| Key | Type | Value |
|---|---|---|
| `overall` | `float` | Correlation across all rows |
| `by_group` | `dict` | Maps each group value to its correlation |
| `reverses` | `bool` | `True` if **any** group's correlation has the opposite sign to `overall` |
| `n_groups` | `int` | Number of groups that produced a correlation |
| `weakest_group` | any | The group whose correlation is smallest in **absolute** value |

**Requirements on behaviour**

- Skip groups with fewer than 3 rows — a correlation from two points is always $\pm 1$ and
  meaningless. They must not appear in `by_group` and must not count towards `n_groups`.
- Skip groups whose correlation is `NaN` (which happens when a column is constant within the
  group).
- Raise `KeyError` if any named column is absent.
- Raise `ValueError` if no group survives the filtering.
- Do not modify the input.

**Hints**

- Iterate the groups with `for name, group in df.groupby(group_col, observed=True):` and build
  the dict yourself. It is clearer than a one-line `apply` here, because you need to skip
  groups conditionally.
- `pd.isna(value)` catches the constant-column case.
- `np.sign()` compares directions. A correlation of exactly 0 has sign 0; treat that as not a
  reversal.
- `min(by_group, key=lambda k: abs(by_group[k]))` finds the weakest.

In [ ]:
# Problem 2 — starter code.

def check_within_groups(df: pd.DataFrame, x_col: str, y_col: str,
                        group_col: str) -> dict:
    """Check whether a correlation survives conditioning on a group.

    Parameters
    ----------
    df : pandas.DataFrame
        The data. Not modified.
    x_col, y_col : str
        The two numeric columns to correlate.
    group_col : str
        The column to condition on.

    Returns
    -------
    dict
        Keys: 'overall', 'by_group', 'reverses', 'n_groups', 'weakest_group'.

    Raises
    ------
    KeyError
        If any named column is absent.
    ValueError
        If no group has enough rows to produce a correlation.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete check_within_groups()")


result = check_within_groups(clean, "attendance_pct", "gpa", "faculty")
print(f"overall  : {result['overall']:+.3f}")
print(f"reverses : {result['reverses']}")
print(f"weakest  : {result['weakest_group']}")
for name, value in sorted(result["by_group"].items()):
    print(f"  {name:<14}{value:+.3f}")

In [ ]:
# Problem 2 — validation. All assertions must pass.

# Built so that the overall correlation is POSITIVE and both groups are
# NEGATIVE — a sign reversal, which is Simpson's paradox in correlation form.
reversal = pd.DataFrame({
    "g": ["low"] * 5 + ["high"] * 5,
    "x": [1.0, 2.0, 3.0, 4.0, 5.0, 11.0, 12.0, 13.0, 14.0, 15.0],
    "y": [10.0, 9.0, 8.0, 7.0, 6.0, 30.0, 29.0, 28.0, 27.0, 26.0],
})

out = check_within_groups(reversal, "x", "y", "g")

assert isinstance(out, dict), "check_within_groups must return a dict"
expected_keys = {"overall", "by_group", "reverses", "n_groups", "weakest_group"}
assert set(out.keys()) == expected_keys, (
    f"Keys must be exactly {sorted(expected_keys)}; got {sorted(out.keys())}"
)

# --- the reversal is detected ---
assert out["overall"] > 0, f"The overall correlation should be positive; got {out['overall']}"
assert all(v < 0 for v in out["by_group"].values()), (
    f"Both groups should be negative; got {out['by_group']}"
)
assert out["reverses"] is True, (
    "Positive overall and negative in every group is a reversal — this is "
    "Simpson's paradox expressed as a correlation"
)
assert isinstance(out["reverses"], bool), "reverses must be a Python bool"

assert out["n_groups"] == 2
assert set(out["by_group"]) == {"low", "high"}
assert isinstance(out["overall"], float)

# --- a case with no reversal ---
steady = pd.DataFrame({
    "g": ["a"] * 5 + ["b"] * 5,
    "x": [1.0, 2.0, 3.0, 4.0, 5.0, 1.0, 2.0, 3.0, 4.0, 5.0],
    "y": [2.0, 4.0, 6.0, 8.0, 10.0, 3.0, 5.0, 7.0, 9.0, 11.0],
})
same = check_within_groups(steady, "x", "y", "g")
assert same["reverses"] is False, "Both groups agree with the overall direction"
assert same["n_groups"] == 2

# --- tiny groups are skipped ---
with_tiny = pd.DataFrame({
    "g": ["big"] * 6 + ["tiny"] * 2,
    "x": [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 1.0, 2.0],
    "y": [2.0, 4.0, 5.0, 8.0, 9.0, 12.0, 5.0, 1.0],
})
skipped = check_within_groups(with_tiny, "x", "y", "g")
assert "tiny" not in skipped["by_group"], (
    "A group of 2 rows gives a correlation of exactly +/-1 and is meaningless. "
    "Skip groups with fewer than 3 rows."
)
assert skipped["n_groups"] == 1

# --- a constant column inside a group gives NaN and is skipped ---
with_constant = pd.DataFrame({
    "g": ["ok"] * 4 + ["flat"] * 4,
    "x": [1.0, 2.0, 3.0, 4.0, 5.0, 5.0, 5.0, 5.0],
    "y": [2.0, 4.0, 6.0, 8.0, 1.0, 2.0, 3.0, 4.0],
})
handled = check_within_groups(with_constant, "x", "y", "g")
assert "flat" not in handled["by_group"], (
    "x is constant within group 'flat', so its correlation is NaN. Skip it."
)
assert not any(pd.isna(v) for v in handled["by_group"].values()), (
    "No NaN may appear in by_group"
)

# --- weakest_group ---
assert out["weakest_group"] in out["by_group"], "weakest_group must be one of the groups"
weakest_value = min(abs(v) for v in out["by_group"].values())
assert np.isclose(abs(out["by_group"][out["weakest_group"]]), weakest_value)

# --- errors ---
for bad in [("nope", "y", "g"), ("x", "nope", "g"), ("x", "y", "nope")]:
    try:
        check_within_groups(reversal, *bad)
    except KeyError:
        pass
    else:
        raise AssertionError(f"Missing column {bad} must raise KeyError")

all_tiny = pd.DataFrame({"g": ["a", "a", "b", "b"],
                         "x": [1.0, 2.0, 3.0, 4.0],
                         "y": [1.0, 2.0, 3.0, 4.0]})
try:
    check_within_groups(all_tiny, "x", "y", "g")
except ValueError:
    pass
else:
    raise AssertionError("When every group is too small, raise ValueError")

# --- the input is untouched ---
before = reversal.copy()
_ = check_within_groups(reversal, "x", "y", "g")
assert reversal.equals(before), "The function must not modify its input"

# --- it works on the notebook's own data ---
real = check_within_groups(clean, "attendance_pct", "gpa", "faculty")
assert real["n_groups"] == 4
assert real["reverses"] is False, "Attendance and GPA agree in every faculty here"

print("Problem 2: all assertions passed.")

---

## Written question

Answer in the Markdown cell below, in 150–250 words.

Section 2.5 showed admission rates that favoured type X overall and type Y in every single
department.

1. Explain in your own words why both tables can be correct at once.
2. A colleague says "the aggregate table is simply wrong, so we should ignore it". Say why
   that is not quite right, and give a question the aggregate table *does* correctly answer.
3. You are analysing whether a new teaching method improved exam scores. Name one grouping
   variable you would check before reporting an overall effect, and say what you would
   conclude if the effect reversed within those groups.

*Replace this text with your answer.*

**1.**

**2.**

**3.**

---

## Submission checklist

- [ ] All four workshop tasks print their `complete` message.
- [ ] Problem 1 prints `all assertions passed`.
- [ ] Problem 2 prints `all assertions passed`.
- [ ] The written question is answered in 150–250 words.
- [ ] **Restart Kernel and Run All Cells** completes with no errors.
- [ ] Committed and pushed to your `puc-data-analysis` repository.

## Looking ahead to Week 10

Week 10 is time series: dates and times, resampling to a different frequency, and rolling
windows. The `moving_average` function you wrote by hand in Week 3 becomes one method call.

Before the session, read McKinney §11.1–11.3.

---

## References

McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media. §5.3, §7.2.

Pfaffelhuber, P. *Python for Data Analysis* (course notes). §9.1.

Bickel, P. J., Hammel, E. A., & O'Connell, J. W. (1975). Sex bias in graduate admissions:
Data from Berkeley. *Science*, 187(4175), 398–404.

Simpson, E. H. (1951). The interpretation of interaction in contingency tables. *Journal of
the Royal Statistical Society: Series B*, 13(2), 238–241.